# Первые модели

В первом ноутбуке бейслайн дал P@R>=70% около 0.389
Теперь собираю нормальную функцию признаков для train и test и смотрю, насколько можно поднять качество без слишком сложных идей

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from metric import precision_at_recall

print(PROJECT_ROOT)

/Users/an.m.titova/Documents/bot_detection_case


In [2]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

events = pd.read_csv(
    DATA_DIR / "events.csv.gz",
    parse_dates=["event_ts"],
)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)

train: (11091, 5)
test: (4909, 4)
events: (328905, 14)


In [3]:
def filter_events_by_window(events, meta):
    events_with_window = events.merge(
        meta[
            [
                "cookie_id",
                "window_start_ts",
                "window_end_ts",
            ]
        ],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )

    mask = (
        (events_with_window["event_ts"] >=
         events_with_window["window_start_ts"])
        &
        (events_with_window["event_ts"] <
         events_with_window["window_end_ts"])
    )

    return events_with_window.loc[
        mask,
        events.columns
    ].copy()

events_train = filter_events_by_window(events, train)
events_test = filter_events_by_window(events, test)

print("Train events:", len(events_train))
print("Test events:", len(events_test))

Train events: 198436
Test events: 89690


In [4]:
def prepare_events(events):
    result = events.copy()

    result["platform_clean"] = (
        result["platform"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return result

events_train = prepare_events(events_train)
events_test = prepare_events(events_test)

EVENT_TYPES = sorted(
    events_train["event_name"]
    .dropna()
    .unique()
)

PLATFORM_TYPES = sorted(
    events_train["platform_clean"]
    .dropna()
    .unique()
)

print(EVENT_TYPES)
print(PLATFORM_TYPES)

['contact_chat_open', 'contact_message_sent', 'contact_phone_show', 'favorite_add', 'item_view', 'login', 'photo_swipe', 'search_results_view', 'seller_page_view']
['android', 'desktop', 'ios', 'iphone', 'web']


Сразу фильтрую events по окну каждой cookie. Это одна из главных проверок во всем решении, модель не должна видеть события после `window_end_ts`

In [5]:
def build_features(meta, events, event_types, platform_types):
    events = events.copy()
    events["platform_clean"] = (
        events["platform"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    features = meta[
        [
            "cookie_id",
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()



    features["cookie_age_days"] = (
        features["window_start_ts"]
        - features["cookie_created_at"]
    ).dt.total_seconds() / (24 * 60 * 60)

    

    event_counts = pd.crosstab(
        events["cookie_id"],
        events["event_name"],
    )

    event_counts = (
        event_counts
        .reindex(columns=event_types, fill_value=0)
        .add_prefix("cnt_event_")
        .reset_index()
    )

    features = features.merge(
        event_counts,
        on="cookie_id",
        how="left",
    )

    count_cols = [
        f"cnt_event_{event_type}"
        for event_type in event_types
    ]

    features[count_cols] = (
        features[count_cols]
        .fillna(0)
    )

    features["n_events"] = (
        features[count_cols]
        .sum(axis=1)
    )

    features["n_event_types"] = (
        (features[count_cols] > 0)
        .sum(axis=1)
    )

   

    for col in count_cols:
        event_name = col.replace("cnt_event_", "")

        features[f"share_event_{event_name}"] = (
            features[col]
            / features["n_events"].replace(0, np.nan)
        )

        features[f"has_event_{event_name}"] = (
            features[col] > 0
        ).astype(int)

    

    diversity = (
        events
        .groupby("cookie_id")
        .agg(
            n_unique_items=("item_id", "nunique"),
            n_unique_categories=("item_category", "nunique"),
            n_unique_locations=("item_location", "nunique"),
            n_unique_seller_types=("seller_type", "nunique"),
            n_unique_queries=("search_query", "nunique"),
            n_unique_platforms=("platform_clean", "nunique"),
            n_unique_user_agents=("user_agent", "nunique"),
        )
        .reset_index()
    )

    features = features.merge(
        diversity,
        on="cookie_id",
        how="left",
    )

    diversity_cols = [
        "n_unique_items",
        "n_unique_categories",
        "n_unique_locations",
        "n_unique_seller_types",
        "n_unique_queries",
        "n_unique_platforms",
        "n_unique_user_agents",
    ]

    features[diversity_cols] = (
        features[diversity_cols]
        .fillna(0)
    )



    search_features = (
        events
        .groupby("cookie_id")
        .agg(
            mean_search_page=("search_page", "mean"),
            max_search_page=("search_page", "max"),
        )
        .reset_index()
    )

    features = features.merge(
        search_features,
        on="cookie_id",
        how="left",
    )



    pointer = events[
        [
            "cookie_id",
            "pointer_x",
            "pointer_y",
        ]
    ].copy()

    pointer["has_pointer"] = (
        pointer["pointer_x"].notna()
        & pointer["pointer_y"].notna()
    )

    pointer_features = (
        pointer
        .groupby("cookie_id")
        .agg(
            n_pointer_events=("has_pointer", "sum"),
            share_pointer_events=("has_pointer", "mean"),
        )
        .reset_index()
    )

    features = features.merge(
        pointer_features,
        on="cookie_id",
        how="left",
    )

    features[
        [
            "n_pointer_events",
            "share_pointer_events",
        ]
    ] = features[
        [
            "n_pointer_events",
            "share_pointer_events",
        ]
    ].fillna(0)



    platform_counts = pd.crosstab(
        events["cookie_id"],
        events["platform_clean"],
    )

    platform_counts = (
        platform_counts
        .reindex(columns=platform_types, fill_value=0)
        .add_prefix("cnt_platform_")
        .reset_index()
    )

    features = features.merge(
        platform_counts,
        on="cookie_id",
        how="left",
    )

    platform_cols = [
        f"cnt_platform_{platform}"
        for platform in platform_types
    ]

    features[platform_cols] = (
        features[platform_cols]
        .fillna(0)
    )

    for col in platform_cols:
        platform = col.replace("cnt_platform_", "")

        features[f"share_platform_{platform}"] = (
            features[col]
            / features["n_events"].replace(0, np.nan)
        )

    
    headless = events[
        ["cookie_id", "user_agent"]
    ].copy()

    headless["is_headless"] = (
        headless["user_agent"]
        .fillna("")
        .str.lower()
        .str.contains(
            "headlesschrome",
            regex=False,
        )
        .astype(int)
    )

    headless_features = (
        headless
        .groupby("cookie_id")
        .agg(
            n_headless_events=("is_headless", "sum"),
            share_headless_events=("is_headless", "mean"),
            has_headless=("is_headless", "max"),
        )
        .reset_index()
    )

    features = features.merge(
        headless_features,
        on="cookie_id",
        how="left",
    )

    features[
        [
            "n_headless_events",
            "share_headless_events",
            "has_headless",
        ]
    ] = features[
        [
            "n_headless_events",
            "share_headless_events",
            "has_headless",
        ]
    ].fillna(0)



    events_sorted = (
        events
        .sort_values(
            [
                "cookie_id",
                "event_ts",
            ]
        )
        .copy()
    )

    events_sorted["time_diff_sec"] = (
        events_sorted
        .groupby("cookie_id")["event_ts"]
        .diff()
        .dt.total_seconds()
    )

    time_features = (
        events_sorted
        .groupby("cookie_id")["time_diff_sec"]
        .agg(
            mean_gap_sec="mean",
            median_gap_sec="median",
            min_gap_sec="min",
            max_gap_sec="max",
            std_gap_sec="std",
        )
        .reset_index()
    )


    valid_gaps = events_sorted[
        events_sorted["time_diff_sec"].notna()
    ].copy()

    valid_gaps["gap_le_1s"] = (
        valid_gaps["time_diff_sec"] <= 1
    )

    valid_gaps["gap_le_5s"] = (
        valid_gaps["time_diff_sec"] <= 5
    )

    valid_gaps["gap_le_10s"] = (
        valid_gaps["time_diff_sec"] <= 10
    )

    fast_actions = (
        valid_gaps
        .groupby("cookie_id")
        .agg(
            share_gap_le_1s=("gap_le_1s", "mean"),
            share_gap_le_5s=("gap_le_5s", "mean"),
            share_gap_le_10s=("gap_le_10s", "mean"),
        )
        .reset_index()
    )

   
    activity_span = (
        events_sorted
        .groupby("cookie_id")["event_ts"]
        .agg(
            first_event_ts="min",
            last_event_ts="max",
        )
        .reset_index()
    )

    activity_span["active_span_sec"] = (
        activity_span["last_event_ts"]
        - activity_span["first_event_ts"]
    ).dt.total_seconds()

    time_features = (
        time_features
        .merge(
            fast_actions,
            on="cookie_id",
            how="left",
        )
        .merge(
            activity_span[
                [
                    "cookie_id",
                    "active_span_sec",
                ]
            ],
            on="cookie_id",
            how="left",
        )
    )

    features = features.merge(
        time_features,
        on="cookie_id",
        how="left",
    )



    def safe_ratio(num, den):
        return (
            num
            / den.replace(0, np.nan)
        )

    if "cnt_event_item_view" in features.columns:
        features["unique_items_per_item_view"] = (
            safe_ratio(
                features["n_unique_items"],
                features["cnt_event_item_view"],
            )
        )

    if (
        "cnt_event_photo_swipe" in features.columns
        and "cnt_event_item_view" in features.columns
    ):
        features["photo_swipe_per_item_view"] = (
            safe_ratio(
                features["cnt_event_photo_swipe"],
                features["cnt_event_item_view"],
            )
        )

    if (
        "cnt_event_favorite_add" in features.columns
        and "cnt_event_item_view" in features.columns
    ):
        features["favorite_per_item_view"] = (
            safe_ratio(
                features["cnt_event_favorite_add"],
                features["cnt_event_item_view"],
            )
        )




    assert features["cookie_id"].is_unique

    assert not any(
        col.endswith("_x")
        or col.endswith("_y")
        for col in features.columns
    )

    return features

In [6]:
train_features = build_features(
    train,
    events_train,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

test_features = build_features(
    test,
    events_test,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

train_features = train_features.merge(
    train[["cookie_id", "target"]],
    on="cookie_id",
    how="left",
)

assert not any(
    col.endswith("_x") or col.endswith("_y")
    for col in train_features.columns
)

print("Количество признаков:", train_features.shape[1])

Количество признаков: 71


In [7]:
train_features.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,cookie_age_days,cnt_event_contact_chat_open,cnt_event_contact_message_sent,cnt_event_contact_phone_show,cnt_event_favorite_add,cnt_event_item_view,...,max_gap_sec,std_gap_sec,share_gap_le_1s,share_gap_le_5s,share_gap_le_10s,active_span_sec,unique_items_per_item_view,photo_swipe_per_item_view,favorite_per_item_view,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,135.603692,0,0,0,0,3,...,84.0,29.949402,0.166667,0.166667,0.333333,167.0,1.333333,0.333333,0.000000,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,194.576111,0,0,1,1,22,...,8511.0,2416.286861,0.000000,0.075000,0.200000,39476.0,0.863636,0.181818,0.045455,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,32.994421,1,1,2,3,7,...,8350.0,2306.449400,0.028571,0.114286,0.171429,36491.0,2.714286,0.428571,0.428571,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0.546759,1,0,1,0,14,...,8604.0,2502.081501,0.000000,0.000000,0.050000,17221.0,0.714286,0.000000,0.000000,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,94.837095,0,0,1,2,13,...,7224.0,1720.954514,0.035714,0.035714,0.035714,19098.0,1.230769,0.307692,0.153846,0


In [8]:
VALID_START = pd.Timestamp("2026-04-17")

is_valid = (
    train_features["window_start_ts"]
    >= VALID_START
)

train_part = train_features.loc[~is_valid].copy()
valid_part = train_features.loc[is_valid].copy()

NON_FEATURE_COLS = [
    "cookie_id",
    "target",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

feature_cols = [
    col
    for col in train_features.columns
    if col not in NON_FEATURE_COLS
]

X_train = train_part[feature_cols]
y_train = train_part["target"]

X_valid = valid_part[feature_cols]
y_valid = valid_part["target"]

model_v2 = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=15,
    random_state=RANDOM_STATE,
)

model_v2.fit(X_train, y_train)

valid_score_v2 = model_v2.predict_proba(X_valid)[:, 1]

p_at_r_v2 = precision_at_recall(
    y_valid,
    valid_score_v2,
)

roc_v2 = roc_auc_score(
    y_valid,
    valid_score_v2,
)

pr_v2 = average_precision_score(
    y_valid,
    valid_score_v2,
)

print("P@R>=70%:", round(p_at_r_v2, 4))
print("ROC-AUC:", round(roc_v2, 4))
print("PR-AUC:", round(pr_v2, 4))

P@R>=70%: 0.5657
ROC-AUC: 0.8883
PR-AUC: 0.6894


Ура, уже сильно лучше бейзлайн. HGB на расширенных признаках дает около 0.566 против 0.389 в первой версии, значит направление с агрегатами по событиям и времени работает

In [9]:
print("Все колонки:", train_features.shape[1])
print("Признаки модели:", len(feature_cols))

Все колонки: 71
Признаки модели: 66


In [10]:
forbidden = {
    "cookie_id",
    "target",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
}

assert forbidden.isdisjoint(feature_cols)

Отдельно проверяю, что в модель случайно не попали `cookie_id`, target и даты как сырые поля

In [11]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False,
)

cat_model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=80,
    verbose=False,
)

valid_score_cat = cat_model.predict_proba(X_valid)[:, 1]

cat_p_at_r = precision_at_recall(
    y_valid,
    valid_score_cat,
)

cat_roc = roc_auc_score(
    y_valid,
    valid_score_cat,
)

cat_pr = average_precision_score(
    y_valid,
    valid_score_cat,
)

print("CatBoost")
print("P@R>=70%:", round(cat_p_at_r, 4))
print("ROC-AUC:", round(cat_roc, 4))
print("PR-AUC:", round(cat_pr, 4))

CatBoost
P@R>=70%: 0.5714
ROC-AUC: 0.8913
PR-AUC: 0.7145


In [12]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat_model.get_feature_importance(),
}).sort_values(
    "importance",
    ascending=False,
)

feature_importance.head(25).reset_index(drop=True)

,feature,importance
0,median_gap_sec,12.423117
1,n_unique_categories,8.726952
2,n_unique_locations,7.676872
3,mean_search_page,5.634302
4,unique_items_per_item_view,4.735011
5,cookie_age_days,3.713110
6,min_gap_sec,3.476074
7,n_events,2.492964
8,max_search_page,2.439815
9,share_event_contact_phone_show,2.329118


Самый важный признак сейчас `median_gap_sec`.

Дальше идут категории, локации, глубина поиска, возраст куки и количество уникальных объявлений

In [13]:
events_train["user_agent"].value_counts(dropna=False).head(30)

user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36     2330
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36    2254
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36                  2252
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36     2182
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36     2181
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36     2169
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36                        2168
Mozilla/5

In [14]:
print(
    "Уникальных User-Agent:",
    events_train["user_agent"].nunique()
)

Уникальных User-Agent: 148


In [15]:
ua_cookie_stats = (
    events_train[
        ["cookie_id", "user_agent"]
    ]
    .drop_duplicates()
    .merge(
        train[["cookie_id", "target"]],
        on="cookie_id",
        how="left",
    )
    .groupby("user_agent")
    .agg(
        n_cookies=("cookie_id", "nunique"),
        bot_share=("target", "mean"),
    )
    .sort_values(
        "n_cookies",
        ascending=False,
    )
)

ua_cookie_stats.head(30)

,n_cookies,bot_share
user_agent,,
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",155,0.083871
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36",151,0.092715
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",150,0.060000
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0,145,0.110345
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 YaBrowser/23.1.0.0 Safari/537.36",143,0.041958
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",142,0.077465
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",142,0.063380
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0,142,0.084507
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",141,0.085106


In [16]:
ua_eda = events_train[
    ["cookie_id", "user_agent"]
].copy()

ua = ua_eda["user_agent"].fillna("").str.lower()

ua_eda["ua_family"] = np.select(
    [
        ua.str.contains("headlesschrome", regex=False),
        ua.str.contains("yabrowser", regex=False),
        ua.str.contains("firefox", regex=False),
        ua.str.contains("chrome", regex=False),
        ua.str.contains("safari", regex=False),
    ],
    [
        "headless_chrome",
        "yandex",
        "firefox",
        "chrome",
        "safari",
    ],
    default="other",
)

ua_eda["ua_windows"] = ua.str.contains(
    "windows", regex=False
).astype(int)

ua_eda["ua_mac"] = (
    ua.str.contains("macintosh", regex=False)
).astype(int)

ua_eda["ua_linux"] = (
    ua.str.contains("linux", regex=False)
).astype(int)

In [17]:
ua_family_stats = (
    ua_eda[
        ["cookie_id", "ua_family"]
    ]
    .drop_duplicates()
    .merge(
        train[["cookie_id", "target"]],
        on="cookie_id",
        how="left",
    )
    .groupby("ua_family")
    .agg(
        n_cookies=("cookie_id", "nunique"),
        bot_share=("target", "mean"),
    )
    .sort_values(
        "bot_share",
        ascending=False,
    )
)

ua_family_stats

,n_cookies,bot_share
ua_family,,
headless_chrome,286,0.262238
other,2341,0.102947
firefox,1682,0.083234
yandex,1459,0.080192
chrome,5912,0.062585
safari,192,0.046875


In [18]:
headless_stats = (
    ua_eda
    .assign(
        is_headless=(
            ua_eda["ua_family"] == "headless_chrome"
        ).astype(int)
    )
    .groupby("cookie_id")
    .agg(
        n_headless_events=("is_headless", "sum"),
        share_headless_events=("is_headless", "mean"),
        has_headless=("is_headless", "max"),
    )
    .reset_index()
    .merge(
        train[["cookie_id", "target"]],
        on="cookie_id",
        how="left",
    )
)

headless_stats.groupby("target")[
    [
        "n_headless_events",
        "share_headless_events",
        "has_headless",
    ]
].mean()

,n_headless_events,share_headless_events,has_headless
target,,,
0,0.343014,0.020703,0.020703
1,3.901001,0.083426,0.083426


HeadlessChrome заметно выделяется по доле ботов. Пробовала добавить большой набор UA-признаков, но на temporal validation стало хуже. Поэтому весь UA-блок не оставляю, беру только компактные признаки про HeadlessChrome

In [19]:
def evaluate_temporal_fold(
    valid_start,
    valid_end,
    features,
    feature_cols,
):
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    train_mask = (
        features["window_start_ts"] < valid_start
    )

    valid_mask = (
        (features["window_start_ts"] >= valid_start)
        &
        (features["window_start_ts"] <= valid_end)
    )

    X_train_fold = features.loc[
        train_mask,
        feature_cols,
    ]

    y_train_fold = features.loc[
        train_mask,
        "target",
    ]

    X_valid_fold = features.loc[
        valid_mask,
        feature_cols,
    ]

    y_valid_fold = features.loc[
        valid_mask,
        "target",
    ]

    model = CatBoostClassifier(
        iterations=600,
        depth=6,
        learning_rate=0.03,
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=(X_valid_fold, y_valid_fold),
        early_stopping_rounds=80,
        verbose=False,
    )

    score = model.predict_proba(
        X_valid_fold
    )[:, 1]

    return {
        "train_size": len(X_train_fold),
        "valid_size": len(X_valid_fold),
        "n_bots_valid": int(y_valid_fold.sum()),
        "p_at_r_70": precision_at_recall(
            y_valid_fold,
            score,
        ),
        "roc_auc": roc_auc_score(
            y_valid_fold,
            score,
        ),
        "pr_auc": average_precision_score(
            y_valid_fold,
            score,
        ),
        "best_iteration": model.get_best_iteration(),
    }

In [20]:
folds = [
    ("2026-04-11", "2026-04-13"),
    ("2026-04-14", "2026-04-16"),
    ("2026-04-17", "2026-04-19"),
]

results = []

for valid_start, valid_end in folds:
    result = evaluate_temporal_fold(
        valid_start,
        valid_end,
        train_features,
        feature_cols,
    )

    result["valid_start"] = valid_start
    result["valid_end"] = valid_end

    results.append(result)

cv_results = pd.DataFrame(results)

cv_results

,train_size,valid_size,n_bots_valid,p_at_r_70,roc_auc,pr_auc,best_iteration,valid_start,valid_end
0,4217,2556,199,0.456026,0.877355,0.667808,409,2026-04-11,2026-04-13
1,6773,2367,195,0.458194,0.906882,0.696399,516,2026-04-14,2026-04-16
2,9140,1951,160,0.571429,0.891265,0.714547,594,2026-04-17,2026-04-19


In [21]:
cv_results[
    [
        "p_at_r_70",
        "roc_auc",
        "pr_auc",
    ]
].mean()

p_at_r_70    0.495216
roc_auc      0.891834
pr_auc       0.692918
dtype: float64

In [22]:
cv_results[
    [
        "p_at_r_70",
        "roc_auc",
        "pr_auc",
    ]
].std()

p_at_r_70    0.066011
roc_auc      0.014772
pr_auc       0.023563
dtype: float64

На трех временных кусках средний P@R>=70% около 0.495.

Последний фолд заметно лучше ранних и дает около 0.571. Потому что к концу данных у модели больше истории для обучения.